# Heart Disease Prediction - EDA and Model Development
**MLOps Assignment 01 | AIMLCZG523**

This notebook covers:
1. Data Acquisition from UCI Repository
2. Exploratory Data Analysis (EDA)
3. Feature Engineering & Preprocessing
4. Model Training (Logistic Regression + Random Forest)
5. Experiment Tracking with MLflow
6. Model Evaluation & Selection

## 0. Install & Import Dependencies

In [ ]:
# Install required libraries (run once)
# !pip install pandas numpy matplotlib seaborn scikit-learn mlflow ucimlrepo

import os
import sys
import warnings
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import mlflow
import mlflow.sklearn

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve
)

matplotlib.use('Agg')
warnings.filterwarnings('ignore')

# Ensure reports directory exists
Path('../backend/reports').mkdir(parents=True, exist_ok=True)

print('All imports successful!')
print(f'NumPy: {np.__version__}, Pandas: {pd.__version__}, MLflow: {mlflow.__version__}')

## 1. Data Acquisition

In [ ]:
# Download dataset from UCI ML Repository
DATA_DIR = Path('../backend/data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_FILE = DATA_DIR / 'processed.cleveland.data'

if not RAW_FILE.exists():
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
    print(f'Downloading dataset from UCI...')
    urllib.request.urlretrieve(url, RAW_FILE)
    print(f'Dataset saved to: {RAW_FILE}')
else:
    print(f'Dataset already exists at: {RAW_FILE}')

In [ ]:
# Load raw dataset
COLUMNS = [
    'age', 'sex', 'cp', 'trestbps', 'chol',
    'fbs', 'restecg', 'thalach', 'exang',
    'oldpeak', 'slope', 'ca', 'thal', 'target'
]

df_raw = pd.read_csv(RAW_FILE, names=COLUMNS)
print(f'Dataset shape: {df_raw.shape}')
print(f'\nFirst 5 rows:')
df_raw.head()

## 2. Data Cleaning & Preprocessing

In [ ]:
# Replace '?' with NaN and convert to numeric
df = df_raw.copy()
df.replace('?', np.nan, inplace=True)
df = df.apply(pd.to_numeric, errors='coerce')

print('Missing values before cleaning:')
print(df.isnull().sum())

# Fill missing values with median
df.fillna(df.median(), inplace=True)

# Binarize target (0=No Disease, 1=Disease)
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

print('\nMissing values after cleaning:')
print(df.isnull().sum())
print(f'\nTarget distribution:')
print(df['target'].value_counts())

In [ ]:
# Save cleaned dataset
CLEAN_DIR = Path('../backend/data/processed')
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_DIR / 'heart_disease_cleaned.csv', index=False)
print('Cleaned dataset saved!')
df.describe()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# --- Class Balance ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
counts = df['target'].value_counts()
axes[0].bar(['No Disease (0)', 'Disease (1)'], counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution (Heart Disease)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=['No Disease', 'Disease'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Target Proportions', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../backend/reports/class_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Class balance: fairly balanced dataset')

In [ ]:
# --- Correlation Heatmap ---
plt.figure(figsize=(12, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, center=0, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../backend/reports/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key finding: cp, thalach, exang show strong correlation with target')

In [ ]:
# --- Feature Distributions (Histograms) ---
num_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].hist(df[df['target']==0][col], bins=20, alpha=0.7,
                 color='#2ecc71', label='No Disease', edgecolor='white')
    axes[i].hist(df[df['target']==1][col], bins=20, alpha=0.7,
                 color='#e74c3c', label='Disease', edgecolor='white')
    axes[i].set_title(col.upper(), fontweight='bold')
    axes[i].legend()
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

axes[-1].axis('off')  # hide last empty subplot
plt.suptitle('Feature Distributions by Target Class', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../backend/reports/numerical_features_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Age vs Max Heart Rate (key insight) ---
plt.figure(figsize=(10, 6))
scatter = plt.scatter(df['age'], df['thalach'],
                      c=df['target'], cmap='RdYlGn_r',
                      alpha=0.7, s=60, edgecolors='white', linewidth=0.5)
plt.colorbar(scatter, label='Heart Disease (1=Yes)')
plt.xlabel('Age', fontsize=12)
plt.ylabel('Max Heart Rate Achieved (thalach)', fontsize=12)
plt.title('Age vs Max Heart Rate - Colored by Disease Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../backend/reports/age_vs_thalach.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Younger patients with higher max heart rate tend to have no disease')

## 4. Feature Engineering

In [ ]:
# Define feature groups
TARGET = 'target'
NUM_FEATURES = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
CAT_FEATURES = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

X = df.drop(TARGET, axis=1)
y = df[TARGET]

# Preprocessing pipeline
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, NUM_FEATURES),
    ('cat', cat_pipeline, CAT_FEATURES)
])

# Stratified train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')
print(f'Train target balance: {y_train.value_counts().to_dict()}')

## 5. Model Training with MLflow Experiment Tracking

In [ ]:
# Setup MLflow
mlflow.set_tracking_uri('file:///tmp/mlruns_notebook')
mlflow.set_experiment('Heart Disease Prediction - Notebook')

MODELS = {
    'logistic_regression': LogisticRegression(max_iter=1000, random_state=42),
    'random_forest': RandomForestClassifier(n_estimators=200, random_state=42)
}

results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in MODELS.items():
    print(f'\n=== Training: {name} ===')
    with mlflow.start_run(run_name=name):
        pipeline = Pipeline([('preprocess', preprocessor), ('model', model)])
        pipeline.fit(X_train, y_train)

        # Log model and params
        mlflow.sklearn.log_model(pipeline, 'model')
        mlflow.log_params(model.get_params())

        # Cross-validation
        cv_scores = cross_validate(
            pipeline, X_train, y_train, cv=cv,
            scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
        )
        cv_metrics = {f'cv_{m}_mean': cv_scores[f'test_{m}'].mean()
                      for m in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']}
        mlflow.log_metrics(cv_metrics)

        # Test evaluation
        y_pred = pipeline.predict(X_test)
        test_metrics = {
            'test_accuracy': accuracy_score(y_test, y_pred),
            'test_precision': precision_score(y_test, y_pred),
            'test_recall': recall_score(y_test, y_pred),
            'test_f1': f1_score(y_test, y_pred),
            'test_roc_auc': roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1])
        }
        mlflow.log_metrics(test_metrics)

        results.append({'model': name, **cv_metrics, **test_metrics})
        print(f'  CV ROC-AUC: {cv_metrics["cv_roc_auc_mean"]:.4f}')
        print(f'  Test Accuracy: {test_metrics["test_accuracy"]:.4f}')
        print(f'  Test ROC-AUC: {test_metrics["test_roc_auc"]:.4f}')

print('\nAll models trained and logged to MLflow!')

## 6. Model Evaluation & Comparison

In [ ]:
# Results comparison table
results_df = pd.DataFrame(results)
print('=== Model Comparison ===')
results_df[['model', 'test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc']]

In [ ]:
# --- ROC Curves for both models ---
plt.figure(figsize=(10, 7))
colors = ['#3498db', '#e74c3c']

for i, (name, model) in enumerate(MODELS.items()):
    pipeline = Pipeline([('preprocess', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, color=colors[i], lw=2.5,
             label=f'{name.replace("_", " ").title()} (AUC = {auc:.3f})')

plt.plot([0,1], [0,1], 'k--', lw=1.5, alpha=0.7, label='Random Classifier')
plt.fill_between([0,1], [0,1], alpha=0.05, color='grey')
plt.xlabel('False Positive Rate', fontsize=13)
plt.ylabel('True Positive Rate', fontsize=13)
plt.title('ROC Curves - Model Comparison', fontsize=15, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../backend/reports/roc_curves_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Confusion Matrices ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, model) in zip(axes, MODELS.items()):
    pipeline = Pipeline([('preprocess', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)
    cm = confusion_matrix(y_test, pipeline.predict(X_test))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Disease', 'Disease'],
                yticklabels=['No Disease', 'Disease'],
                linewidths=1, cbar=False)
    ax.set_title(f'{name.replace("_", " ").title()}', fontweight='bold', fontsize=13)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('Actual', fontsize=11)

plt.suptitle('Confusion Matrices', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../backend/reports/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Feature Importance (Random Forest) ---
rf_pipeline = Pipeline([('preprocess', preprocessor),
                         ('model', RandomForestClassifier(n_estimators=200, random_state=42))])
rf_pipeline.fit(X_train, y_train)

# Get feature names after encoding
cat_encoder = rf_pipeline.named_steps['preprocess'].named_transformers_['cat']['encoder']
cat_feat_names = cat_encoder.get_feature_names_out(CAT_FEATURES).tolist()
all_feat_names = NUM_FEATURES + cat_feat_names

importances = rf_pipeline.named_steps['model'].feature_importances_
feat_df = pd.DataFrame({'feature': all_feat_names, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=True).tail(15)

plt.figure(figsize=(10, 8))
bars = plt.barh(feat_df['feature'], feat_df['importance'],
                color=plt.cm.RdYlGn(feat_df['importance'] / feat_df['importance'].max()),
                edgecolor='white')
plt.xlabel('Feature Importance', fontsize=12)
plt.title('Top 15 Feature Importances - Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../backend/reports/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Best Model Selection & Summary

In [ ]:
best = results_df.sort_values('test_roc_auc', ascending=False).iloc[0]
print('='*50)
print(f'BEST MODEL: {best["model"].replace("_", " ").upper()}')
print('='*50)
print(f'  Test Accuracy : {best["test_accuracy"]:.4f}')
print(f'  Test Precision: {best["test_precision"]:.4f}')
print(f'  Test Recall   : {best["test_recall"]:.4f}')
print(f'  Test F1 Score : {best["test_f1"]:.4f}')
print(f'  Test ROC-AUC  : {best["test_roc_auc"]:.4f}')
print(f'  CV ROC-AUC    : {best["cv_roc_auc_mean"]:.4f}')
print('='*50)
print('Model is exported to backend/exported_model/ via main_pipeline.py')
print('MLflow tracking URI: file:///tmp/mlruns_notebook')
print('Run `mlflow ui` to view experiment dashboard')